In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
import os
import glob
from torch.utils.data import Dataset
from PIL import Image
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms

# transformations
train_transform = transforms.Compose([
    transforms.Resize((32 , 32)),  # Resize images
    transforms.RandomRotation(15),  # Rotate images randomly within ±15 degrees
    transforms.ToTensor(),  # Convert to tensor
])

# Validation and testing data typically don’t require augmentations, as we only evaluate the model performance on these sets.
# Instead, we apply basic transformations to prepare the images.
test_transform = transforms.Compose([
    transforms.Resize((32 , 32)),  # Resize images to 64x64
    transforms.ToTensor(),  # Convert to tensor
])

#after inspection dataset appears to be organized so we use Imagefolder
path = os.path.join(path, "PlantVillage") #true data path

train_dir = os.path.join(path, "train")
test_dir = os.path.join(path, "test")

train_dataset = ImageFolder(root=train_dir, transform=train_transform)
test_dataset  = ImageFolder(root=test_dir,  transform=test_transform)

#DataLoaders
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=67, shuffle=True) #bonus plz
test_loader  = DataLoader(test_dataset,  batch_size=67, shuffle=False)

print(train_dataset)


In [ ]:
# Write your code here
from torch import nn

class PotatoDisease(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            #first layer
            nn.Conv2d(3, 32, kernel_size=5),  # [B,32,28,28]
            nn.BatchNorm2d(32),
            nn.ReLU(),

            #second
            nn.Conv2d(32, 64, kernel_size=5),  # [B,64,24,24]
            nn.BatchNorm2d(64),
            nn.ReLU(),

            #third
            nn.Conv2d(64, 128, kernel_size=5),  # [B,128,20,20]
            nn.BatchNorm2d(128),
            nn.ReLU(),

            #fourth
            nn.Conv2d(128, 256, kernel_size=5),  # [B,256,16,16]
            nn.BatchNorm2d(256),
            nn.ReLU(),

            #fifth
            nn.Conv2d(256, 512, kernel_size=5,stride=2),  # [B,512,6,6]
            nn.BatchNorm2d(512),
            nn.ReLU(),

        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 6 * 6, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # TO-DO: Pass x through features
        x = self.features(x)
        # TO-DO: Pass result through classifier
        x = self.classifier(x)
        return x


In [ ]:
# Write your code here

#move to gpu if available
import torch
from tqdm import tqdm  #tqdm for progress bar

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PotatoDisease().to(device)

def accuracy_from_logits(logits, labels):
    # TO-DO: Get predicted class indices
    # HINT: Use torch.argmax with dim=1
    preds = torch.argmax(logits, dim=1)
    # TO-DO: Calculate and return accuracy
    # HINT: Compare preds with labels, convert to float, mean, then .item()
    return (preds == labels).float().mean().item()

#training loop
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_acc = 0.0, 0.0

    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)

        # TO-DO: Zero the gradients
        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        # TO-DO: Backward pass
        loss.backward()

        # TO-DO: Update parameters
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits.detach(), labels)

    return total_loss / len(loader), total_acc / len(loader)


#validation loop
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_acc = 0.0, 0.0

    with torch.no_grad():
        for images , labels in tqdm(loader):
            images, labels = images.to(device), labels.to(device)

            # TO-DO: Get model predictions
            logits = model(images)

            # TO-DO: Calculate loss
            # HINT: Use the criterion function
            loss = criterion(logits, labels)

            total_loss += loss.item()
            total_acc += accuracy_from_logits(logits, labels)

    return total_loss / len(loader), total_acc / len(loader)


In [ ]:
# Write your code here



# Training setup
# cross entropy criterion because multi class
criterion = nn.CrossEntropyLoss()

# Set learning rate
learning_rate = 0.001

# Define optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr= learning_rate)

#Set number of epochs
num_epochs = 10

# Initialize history tracking
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

# Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    # Store history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')

In [ ]:
# Write your code here
